In [13]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

In [14]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.167243264


In [15]:
DEV_PROCESSED  = "/content/development_processed.csv"
EVAL_PROCESSED = "/content/evaluation_processed.csv"

df_dev  = pd.read_csv(DEV_PROCESSED)
df_eval = pd.read_csv(EVAL_PROCESSED)
MAX_LEN = 256
print(df_dev.shape, df_eval.shape)

(79997, 12) (20000, 11)


In [16]:
def build_transformer_text(df):
    return (
        df["title"].astype(str) +
        "\n\n" +
        df["article"].astype(str)
    )

df_dev["tr_text"]  = build_transformer_text(df_dev)
df_eval["tr_text"] = build_transformer_text(df_eval)


In [17]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_dev,
    test_size=0.15,
    stratify=df_dev["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)


Train: (67997, 13)
Val: (12000, 13)


In [18]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LEN):
        self.texts  = df["tr_text"].tolist()
        self.labels = df["label"].values if "label" in df else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

In [19]:
from transformers import RobertaTokenizer

MODEL_NAME = "roberta-large"
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

In [20]:
train_ds = NewsDataset(train_df, tokenizer)
val_ds   = NewsDataset(val_df, tokenizer)
eval_ds  = NewsDataset(df_eval, tokenizer)

In [21]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=7
).cuda()

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./roberta_out",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_steps=100,
    report_to="none",
    save_total_limit=2
)


In [23]:
from transformers import Trainer, TrainingArguments


In [24]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro")
    }


In [25]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.719300,0.669058,0.728445
2,0.570800,0.644919,0.741778
3,0.468100,0.665337,0.749052


TrainOutput(global_step=25500, training_loss=0.6318186448041131, metrics={'train_runtime': 2568.4602, 'train_samples_per_second': 79.422, 'train_steps_per_second': 9.928, 'total_flos': 9.505441118547302e+16, 'train_loss': 0.6318186448041131, 'epoch': 3.0})

In [27]:
load_best_model_at_end=True
metric_for_best_model="macro_f1"

In [28]:
full_train_df = df_dev.copy()

full_train_ds = NewsDataset(
    full_train_df,
    tokenizer,
    max_length=MAX_LEN
)

In [29]:
from transformers import TrainingArguments

args_full = TrainingArguments(
    output_dir="./roberta_full_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    report_to="none"
)


In [30]:
trainer_full = Trainer(
    model=trainer.model,
    args=args_full,
    train_dataset=full_train_ds
)


In [31]:
trainer_full.train()

Step,Training Loss
100,0.502700
200,0.503000
300,0.503700
400,0.527500
500,0.499300
600,0.509200
700,0.526000
800,0.487200
900,0.505400
1000,0.519500


TrainOutput(global_step=7500, training_loss=0.4240354398091634, metrics={'train_runtime': 1581.74, 'train_samples_per_second': 151.726, 'train_steps_per_second': 4.742, 'total_flos': 1.1182945911737702e+17, 'train_loss': 0.4240354398091634, 'epoch': 3.0})

In [32]:
preds = trainer_full.predict(eval_ds)
logits = preds.predictions
final_pred = logits.argmax(axis=1)

In [33]:
pred_out = trainer.predict(eval_ds)
logits = pred_out.predictions

np.save(f"logits_roberta_processed_noseed_MAXLEN_{MAX_LEN}.npy", logits)

preds = logits.argmax(axis=1)

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": preds.astype(int)
})

submission.to_csv(f"submission_roberta_processed_noseed_MAXLEN_{MAX_LEN}.csv", index=False)

print(f"Saved submission_noseed_MAX_LEN{MAX_LEN}.csv")
print(f"Saved logits_noseed_MAX_LEN{MAX_LEN}.npy")

Saved submission_noseed_MAX_LEN256.csv
Saved logits_noseed_MAX_LEN256.npy
